# Analysis 2 - Spatial and Accessibility Analysis
**Course**: Big Data Engineering
**Description**: Supporting spatial tasks for Analysis 2 including Geocoding, GeoPandas Choropleth Mapping of SCGI & K-Means Clusters, and OSRM Driving Route Accessibility analysis.

### 0. Dynamic Package Installation
This cell ensures that all necessary spatial packages are installed in the active kernel's Python environment.

In [1]:
import sys
import subprocess

required_packages = ["geopandas", "geopy", "folium", "shapely", "requests", "branca"]
missing_packages = []

for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        missing_packages.append(pkg)

if missing_packages:
    print(f"Missing packages found: {missing_packages}. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_packages)
    print("Installation complete! Please restart the kernel (Kernel -> Restart) if imports still fail.")
else:
    print("All packages are already installed!")

All packages are already installed!


### 1. Import Libraries
We import spatial packages: `geopandas`, `geopy`, `folium`, and `requests` for OSRM API calls.

In [2]:
import os
import json
import requests
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import branca.colormap as cm
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

print("All packages imported successfully!")

All packages imported successfully!


### 2. Geocoding Enrichment (Walkthrough)
This section shows how the kecamatan coordinates (latitude, longitude) were retrieved using Geopy's Nominatim geocoder. 
We read from the geocoded CSV coordinates dataset we created.

In [3]:
coords_path = "data/surabaya_kecamatan_coords.csv"
if os.path.exists(coords_path):
    df_coords = pd.read_csv(coords_path)
    display(df_coords.head())
else:
    print("Coords file not found. Run geocoding script first!")

,kecamatan_norm,latitude,longitude,address
0,PABEAN CANTIAN,-7.216768,112.729338,"Kantor Kecamatan Pabean Cantian, Jalan Teluk S..."
1,SAWAHAN,-7.284066,112.716165,"Kantor Kecamatan Sawahan, Jalan Dukuh Kupang T..."
2,TAMBAKSARI,-7.257488,112.755219,"Kantor Kecamatan Tambak Sari, Jalan Mendut, RW..."
3,ASEMROWO,-7.252032,112.715258,"Kantor Kecamatan Asemrowo, Jalan Asem Raya, RW..."
4,SIMOKERTO,-7.243732,112.757919,"Kantor Kecamatan Simokerto, Jalan Tambak Rejo ..."


### 3. Load GeoJSON Boundaries and Join Analytical Data
We load the local GeoJSON containing kecamatan boundaries (constructed via Voronoi diagram on kecamatan centroids for robust offline use). Then, we join the **SCGI** (School Capacity Gap Index) scores and **K-Means Cluster** priorities.

In [4]:
# Load GeoJSON
geojson_path = "data/surabaya_kecamatan_voronoi.geojson"
gdf = gpd.read_file(geojson_path)

# Load SCGI data
scgi_path = "../medallion/exports/scgi.csv"
df_scgi = pd.read_csv(scgi_path)

# Load Cluster data
clusters_path = "../medallion/exports/clusters.csv"
df_clusters = pd.read_csv(clusters_path)

# Merge data
gdf = gdf.merge(df_scgi[['kecamatan_norm', 'scgi_score', 'scgi_category', 'scgi_rank', 'utilisasi_2030', 'deficit_2030']], on='kecamatan_norm')
gdf = gdf.merge(df_clusters[['kecamatan_norm', 'priority_rank', 'priority_label', 'cluster_id']], on='kecamatan_norm')

print(f"Merged GeoDataFrame: {gdf.shape[0]} kecamatan loaded.")
gdf.head(3)

Merged GeoDataFrame: 31 kecamatan loaded.


,kecamatan_norm,latitude,longitude,geometry,scgi_score,scgi_category,scgi_rank,utilisasi_2030,deficit_2030,priority_rank,priority_label,cluster_id
0,PAKAL,-7.239993,112.625446,"POLYGON ((112.67053 -7.19767, 112.66978 -7.199...",13.58,RENDAH,23,90.5,0.0,4,RENDAH,0
1,BENOWO,-7.248778,112.635413,"POLYGON ((112.60328 -7.27519, 112.66978 -7.199...",24.05,RENDAH,14,115.5,1403.0,3,SEDANG,3
2,LAKARSANTRI,-7.304293,112.632989,"POLYGON ((112.60328 -7.27519, 112.62903 -7.276...",9.99,RENDAH,31,66.6,0.0,4,RENDAH,0


### 4. Choropleth Map: School Capacity Gap Index (SCGI)
The SCGI is colored dynamically from light green (safe) to deep red (critical index).

In [5]:
# Center map around Surabaya
m_scgi = folium.Map(location=[-7.27, 112.74], zoom_start=12, tiles="cartodbpositron")

# Linear colormap
colormap_scgi = cm.linear.YlOrRd_09.scale(gdf['scgi_score'].min(), gdf['scgi_score'].max())
colormap_scgi.caption = 'School Capacity Gap Index (SCGI) Score'
colormap_scgi.add_to(m_scgi)

def style_function_scgi(feature):
    score = feature['properties']['scgi_score']
    return {
        'fillOpacity': 0.6,
        'weight': 1.5,
        'color': '#333333',
        'fillColor': colormap_scgi(score) if score is not None else '#FFFFFF'
    }

# Add GeoJson layer
folium.GeoJson(
    gdf,
    name="SCGI Score Choropleth",
    style_function=style_function_scgi,
    tooltip=folium.GeoJsonTooltip(
        fields=['kecamatan_norm', 'scgi_score', 'scgi_category', 'scgi_rank', 'deficit_2030', 'utilisasi_2030'],
        aliases=['Kecamatan:', 'SCGI Score:', 'Kategori:', 'Peringkat SCGI:', 'Defisit (Siswa):', 'Utilisasi %:'],
        localize=True
    )
).add_to(m_scgi)

m_scgi

### 5. Choropleth Map: K-Means Priority Clusters
Colors are assigned by Priority Rank:
- **Priority 1 (KRITIS)**: Red  
- **Priority 2 (TINGGI)**: Orange  
- **Priority 3 (SEDANG)**: Yellow  
- **Priority 4 (RENDAH)**: Green  

In [6]:
m_cluster = folium.Map(location=[-7.27, 112.74], zoom_start=12, tiles="cartodbpositron")

color_map = {
    'KRITIS': '#D32F2F', # Red
    'TINGGI': '#F57C00', # Orange
    'SEDANG': '#FBC02D', # Yellow
    'RENDAH': '#388E3C'  # Green
}

def style_function_cluster(feature):
    label = feature['properties']['priority_label']
    return {
        'fillOpacity': 0.6,
        'weight': 1.5,
        'color': '#333333',
        'fillColor': color_map.get(label, '#B0BEC5')
    }

folium.GeoJson(
    gdf,
    name="K-Means Priority Clusters",
    style_function=style_function_cluster,
    tooltip=folium.GeoJsonTooltip(
        fields=['kecamatan_norm', 'priority_label', 'cluster_id', 'scgi_score'],
        aliases=['Kecamatan:', 'Prioritas:', 'Cluster ID:', 'SCGI Score:'],
        localize=True
    )
).add_to(m_cluster)

# Add legend manually using folium element
legend_html = '''
     <div style="position: fixed; 
     bottom: 50px; left: 50px; width: 150px; height: 120px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white;
     opacity: 0.85;
     padding: 10px;
     ">
     <b>Priority Level</b><br>
     <i class="fa fa-circle" style="color:#D32F2F"></i> Kritis<br>
     <i class="fa fa-circle" style="color:#F57C00"></i> Tinggi<br>
     <i class="fa fa-circle" style="color:#FBC02D"></i> Sedang<br>
     <i class="fa fa-circle" style="color:#388E3C"></i> Rendah<br>
     </div>
     '''
m_cluster.get_root().html.add_child(folium.Element(legend_html))

m_cluster

### 6. Accessibility Analysis (OSRM Routing Proof-of-Concept)
To analyze access, we perform a routing query using OSRM's public API to compute driving paths, distances, and durations between a deficit kecamatan (e.g. `PABEAN CANTIAN`, Kritis) and a safe/surplus kecamatan (e.g. `GENTENG`, Rendah).

In [7]:
# Centroids for route endpoints
start_name = "PABEAN CANTIAN"
end_name = "GENTENG"

start_row = df_coords[df_coords['kecamatan_norm'] == start_name].iloc[0]
end_row = df_coords[df_coords['kecamatan_norm'] == end_name].iloc[0]

lat1, lon1 = start_row['latitude'], start_row['longitude']
lat2, lon2 = end_row['latitude'], end_row['longitude']

osrm_url = f"https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson"
headers = {'User-Agent': 'Mozilla/5.0'}

print(f"Requesting driving route from {start_name} to {end_name} via OSRM...")
try:
    response = requests.get(osrm_url, headers=headers, timeout=10)
    if response.status_code == 200:
        route_data = response.json()
        route = route_data['routes'][0]
        distance_km = route['distance'] / 1000.0
        duration_min = route['duration'] / 60.0

        print(f"Routing Success!")
        print(f"  Distance: {distance_km:.2f} km")
        print(f"  Duration: {duration_min:.2f} mins")

        # Map route geometry
        m_route = folium.Map(location=[(lat1+lat2)/2, (lon1+lon2)/2], zoom_start=13, tiles="cartodbpositron")
        
        # Add markers
        folium.Marker([lat1, lon1], popup=f"Start: {start_name} (Kritis)", icon=folium.Icon(color='red')).add_to(m_route)
        folium.Marker([lat2, lon2], popup=f"End: {end_name} (Rendah)", icon=folium.Icon(color='green')).add_to(m_route)
        
        # Draw route line
        route_geom = route['geometry']
        # GeoJSON geometry coordinates are [lon, lat], folium needs [lat, lon]
        folium_coords = [[coord[1], coord[0]] for coord in route_geom['coordinates']]
        folium.PolyLine(folium_coords, color="blue", weight=5, opacity=0.8, tooltip=f"{distance_km:.2f} km, {duration_min:.2f} min").add_to(m_route)
    else:
        print(f"OSRM Server error {response.status_code}. Using fallback straight line path.")
        raise Exception("Server Error")
except Exception as e:
    print(f"Error fetching route: {e}. Generating fallback straight-line visualization.")
    # Fallback to straight line
    m_route = folium.Map(location=[(lat1+lat2)/2, (lon1+lon2)/2], zoom_start=13, tiles="cartodbpositron")
    folium.Marker([lat1, lon1], popup=f"Start: {start_name}", icon=folium.Icon(color='red')).add_to(m_route)
    folium.Marker([lat2, lon2], popup=f"End: {end_name}", icon=folium.Icon(color='green')).add_to(m_route)
    folium.PolyLine([[lat1, lon1], [lat2, lon2]], color="red", weight=3, dash_array='5, 5', tooltip="Fallback straight line").add_to(m_route)

m_route

Requesting driving route from PABEAN CANTIAN to GENTENG via OSRM...
Routing Success!
  Distance: 9.69 km
  Duration: 9.61 mins
